# EfficientNet-B0 — Tea Leaf Disease Classification
**CSC4093/DSC4213: Deep Learning (2024/25) — Programming Assignment 02**

Transfer learning using EfficientNet-B0 pretrained on ImageNet.

**Improvements over baseline models:**
- EfficientNet-B0 backbone (more efficient than VGG16/ResNet50)
- LR Warmup + Cosine Annealing scheduler
- Early stopping
- Best model checkpoint saving
- Class-weighted loss function
- Test-time augmentation (TTA)
- Grad-CAM visualization

## 1. Install & Import Libraries

In [13]:
# Uncomment if running for the first time
# !pip install grad-cam torchvision torch scikit-learn matplotlib seaborn

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 2. Dataset Configuration

In [14]:

DATA_DIR    = './dataset'   # folder containing your 3 class subfolders
# ─────────────────────────────────────────────────────────────────────────────

SPLIT_DIR   = './dataset_split'  # auto-created — do not change

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

BATCH_SIZE  = 16
NUM_CLASSES = 3
NUM_EPOCHS  = 25
IMG_SIZE    = 224
SEED        = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Config ready.')

Config ready.


## 3. Auto Split Dataset into Train / Val / Test

In [16]:
def split_dataset(data_dir, split_dir, train_r, val_r, seed=42):
    """
    Reads class folders from data_dir and copies images into:
        split_dir/train/<class>/
        split_dir/val/<class>/
        split_dir/test/<class>/
    Skips silently if split already exists.
    """
    if os.path.exists(split_dir):
        print(f'Split folder "{split_dir}" already exists — skipping.')
        return

    random.seed(seed)
    class_names = sorted([
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ])
    print(f'Found classes: {class_names}\n')

    for split in ['train', 'val', 'test']:
        for cls in class_names:
            os.makedirs(os.path.join(split_dir, split, cls), exist_ok=True)

    for cls in class_names:
        cls_path = os.path.join(data_dir, cls)
        images   = sorted(glob.glob(os.path.join(cls_path, '*.*')))
        images   = [f for f in images
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        random.shuffle(images)

        n       = len(images)
        n_train = int(n * train_r)
        n_val   = int(n * val_r)

        splits = {
            'train': images[:n_train],
            'val':   images[n_train:n_train + n_val],
            'test':  images[n_train + n_val:]
        }
        for split_name, files in splits.items():
            for f in files:
                dst = os.path.join(split_dir, split_name, cls, os.path.basename(f))
                shutil.copy2(f, dst)
            print(f'  {cls:25s} → {split_name:5s}: {len(files)} images')

    print('\n✓ Dataset split complete.')


split_dataset(DATA_DIR, SPLIT_DIR, TRAIN_RATIO, VAL_RATIO, SEED)

# Verify
for split in ['train', 'val', 'test']:
    total = sum(
        len(os.listdir(os.path.join(SPLIT_DIR, split, c)))
        for c in os.listdir(os.path.join(SPLIT_DIR, split))
    )
    print(f'{split:6s}: {total} images')

Split folder "./dataset_split" already exists — skipping.
train : 0 images
val   : 0 images
test  : 0 images


## 3. Data Transforms & Loaders

In [12]:
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=train_transforms)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_DIR, 'val'),   transform=val_test_transforms)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'),  transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print(f'Classes: {train_dataset.classes}')

FileNotFoundError: [Errno 2] No such file or directory: './dataset/train'

## 4. Build EfficientNet-B0 Model

In [ ]:
# Load pretrained EfficientNet-B0
model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

# Freeze all feature layers except the last two blocks
for param in model.features[:-2].parameters():
    param.requires_grad = False

# Replace classifier head
in_features = model.classifier[1].in_features  # 1280
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=True),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, NUM_CLASSES)
)

model = model.to(device)

# Count trainable parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)')

## 5. Loss, Optimizer & Scheduler

In [ ]:
# ── Class-weighted loss to handle imbalanced support ──────────────────────────
train_labels = [label for _, label in train_dataset.samples]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
weights_tensor = torch.FloatTensor(class_weights).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.1)
print(f'Class weights: {class_weights.round(3)}')

# ── Differential learning rates ───────────────────────────────────────────────
optimizer = optim.Adam([
    {'params': model.features[-2:].parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(),    'lr': 1e-3}
], weight_decay=1e-4)

# ── LR Warmup (5 epochs) → Cosine Annealing ──────────────────────────────────
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

warmup_scheduler = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=5)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - 5, eta_min=1e-6)
scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[5])

print('Optimizer, scheduler, and loss function configured.')

## 6. Early Stopping & Checkpoint Utilities

In [ ]:
class EarlyStopping:
    """Stops training when validation loss stops improving."""
    def __init__(self, patience=7, min_delta=0.001):
        self.patience  = patience
        self.min_delta = min_delta
        self.counter   = 0
        self.best_loss = float('inf')
        self.triggered = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
        return self.triggered


def save_checkpoint(model, optimizer, epoch, val_acc, path='best_efficientnet_b0.pth'):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_acc': val_acc,
    }, path)
    print(f'  ✓ Checkpoint saved — epoch {epoch+1}, val_acc={val_acc:.4f}')


early_stopping = EarlyStopping(patience=7)
best_val_acc   = 0.0
print('EarlyStopping and checkpoint utilities ready.')

## 7. Training Loop

In [ ]:
history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += images.size(0)
    return total_loss / total, correct / total


print(f'Training EfficientNet-B0 for up to {NUM_EPOCHS} epochs...\n')

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss,   val_acc   = run_epoch(val_loader,   training=False)
    scheduler.step()

    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch [{epoch+1:02d}/{NUM_EPOCHS}] '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | LR: {lr_now:.2e}')

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(model, optimizer, epoch, val_acc)

    # Early stopping
    if early_stopping(val_loss):
        print(f'\nEarly stopping triggered at epoch {epoch+1}')
        break

print(f'\nTraining complete. Best val accuracy: {best_val_acc:.4f}')

## 8. Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_acc']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_ran, history['train_acc'], 'b-o', label='Train')
axes[0].plot(epochs_ran, history['val_acc'],   'r-o', label='Validation')
axes[0].set_title('EfficientNet-B0 — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_ran, history['train_loss'], 'b-o', label='Train')
axes[1].plot(epochs_ran, history['val_loss'],   'r-o', label='Validation')
axes[1].set_title('EfficientNet-B0 — Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('figures/efficientnet_b0_training_curves.png', dpi=150)
plt.show()
print('Training curves saved to figures/')

## 9. Load Best Checkpoint & Evaluate on Test Set

In [ ]:
# Load best saved model
checkpoint = torch.load('best_efficientnet_b0.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']+1} "
      f"(val_acc={checkpoint['val_acc']:.4f})")

# Standard evaluation
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print('\n— Classification Report —')
print(classification_report(
    all_labels, all_preds,
    target_names=train_dataset.classes
))

## 10. Test-Time Augmentation (TTA) Evaluation

In [ ]:
tta_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

tta_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'), transform=tta_transforms)
tta_loader  = DataLoader(tta_dataset, batch_size=BATCH_SIZE, shuffle=False)

N_AUGMENTS = 5
model.eval()
tta_preds_accum = None

for aug_i in range(N_AUGMENTS):
    batch_probs = []
    with torch.no_grad():
        for images, _ in tta_loader:
            images = images.to(device)
            probs  = torch.softmax(model(images), dim=1).cpu().numpy()
            batch_probs.append(probs)
    run_probs = np.concatenate(batch_probs, axis=0)
    tta_preds_accum = run_probs if tta_preds_accum is None else tta_preds_accum + run_probs

tta_final_preds = np.argmax(tta_preds_accum / N_AUGMENTS, axis=1)
tta_labels      = [label for _, label in tta_dataset.samples]

print('— TTA Classification Report —')
print(classification_report(
    tta_labels, tta_final_preds,
    target_names=train_dataset.classes
))

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=train_dataset.classes,
            yticklabels=train_dataset.classes)
plt.title('EfficientNet-B0 — Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.savefig('figures/efficientnet_b0_confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved to figures/')

## 12. Grad-CAM Visualization

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Target layer — last conv block in EfficientNet-B0
target_layer = [model.features[-1]]
cam = GradCAM(model=model, target_layers=target_layer)

# Pick one image per class from the test set
fig, axes = plt.subplots(NUM_CLASSES, 2, figsize=(8, 12))
class_shown = {}

for img_path, label in test_dataset.samples:
    if label in class_shown or len(class_shown) == NUM_CLASSES:
        continue

    # Prepare tensors
    raw_img = np.array(Image.open(img_path).resize((IMG_SIZE, IMG_SIZE))) / 255.0
    input_tensor = val_test_transforms(Image.open(img_path).convert('RGB')).unsqueeze(0).to(device)

    # Compute Grad-CAM
    targets    = [ClassifierOutputTarget(label)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]
    visualization = show_cam_on_image(raw_img.astype(np.float32), grayscale_cam, use_rgb=True)

    row = label
    axes[row][0].imshow(raw_img)
    axes[row][0].set_title(f'Original: {train_dataset.classes[label]}')
    axes[row][0].axis('off')
    axes[row][1].imshow(visualization)
    axes[row][1].set_title(f'Grad-CAM: {train_dataset.classes[label]}')
    axes[row][1].axis('off')

    class_shown[label] = True

plt.suptitle('Grad-CAM — EfficientNet-B0 Feature Focus', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/efficientnet_b0_gradcam.png', dpi=150)
plt.show()
print('Grad-CAM visualization saved to figures/')

## Summary

| Metric | Value |
|---|---|
| Model | EfficientNet-B0 |
| Pretrained | ImageNet |
| Epochs | Up to 25 (early stopping) |
| Scheduler | LR Warmup + CosineAnnealing |
| Loss | CrossEntropy + Label Smoothing 0.1 + Class Weights |
| TTA Passes | 5 |
| Checkpoint | Best validation accuracy |
